In [1]:
# Setup (Imports, API Key, Prompts)
import os
import json
import re
from openai import OpenAI
from tqdm.notebook import tqdm  # ใช้ tqdm.notebook สำหรับ .ipynb
import pandas as pd
import time
from collections import defaultdict
import getpass  # << เพิ่มบรรทัดนี้

print("🚀 Initializing AI Evaluator...")

# --- 1. Setup OpenAI Client ---
# ถ้ายังไม่มี OPENAI_API_KEY ใน env ให้ถามจากผู้ใช้แบบไม่โชว์ตัวอักษร
if "OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"].startswith("sk-"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

if "OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"].startswith("sk-"):
    print("=" * 50)
    print("Error: OPENAI_API_KEY not set correctly.")
    print("=" * 50)
else:
    print("OpenAI API Key loaded.")

client = OpenAI()  # จะอ่าน key จาก env โดยอัตโนมัติ
EVALUATION_MODEL = "gpt-4o" # ใช้ "gpt-4o" ตัวเต็มเพื่อการประเมินที่แม่นยำ
print(f"Evaluation Model set to: {EVALUATION_MODEL}")

# --- 2. Define File Paths (Relative to this notebook) ---
OUR_MODEL_FILE = "../llms_therapist/therapist_simulation_results.jsonl"
BASELINE_MODEL_FILE = "../llms_therapist_base/therapist_simulation_results_BASELINE.jsonl"
print(f"Our Model results path: {OUR_MODEL_FILE}")
print(f"Baseline Model results path: {BASELINE_MODEL_FILE}")

# --- 3. Define Evaluation Prompts (Exactly as user provided) ---
SYSTEM_PROMPT = """You are an expert psychological evaluator specializing in Cognitive Behavioral Therapy (CBT) and therapeutic alliance. Your task is to analyze a full counseling session transcript and score the therapist's performance based on the evaluation frameworks used in the MIRROR paper (arXiv:2504.13211v2).

CRITICAL EVALUATION FOCUS:
- You must assess the therapist's ability to perceive "Emotional Subtext"—the underlying feelings that may not be explicitly stated in the client's words but are hinted at through context, tone, or provided emotional metadata.
- A high-performing therapist should move beyond surface-level reflection and identify "Latent Concerns" (e.g., hidden guilt, worthiness issues, or conflicting emotions) to facilitate deeper discovery.

You must evaluate the entire session as a whole, not just individual turns.
"""

USER_PROMPT = """Please evaluate this therapy session. Pay close attention to how the therapist navigates the gap between what the client SAYS and what the client FEELS

[Session Transcript]
{CONVERSATION_TEXT_GOES_HERE}
[/Session Transcript]

[Evaluation Task]
Analyze the therapist's responses. You must be extremely critical and look for "Therapeutic Depth."
**Special Note for Metadata:** Some sessions include metadata (valence, arousal, dissonance flags). These represent the client's non-verbal emotional cues detected by the system. Evaluate how effectively the therapist uses these cues to uncover the client's underlying emotions.

1. **Therapist Skills Assessment (General Counseling):**
 * **Understanding (Score 0-6):** How well did the therapist interpret the client's concerns and feelings? (0 = Not at all, 6 = Perfectly)
 * **Interpersonal Effectiveness (Score 0-6):** How well did the therapist maintain a supportive and therapeutic relationship? (0 = Not at all, 6 = Perfectly)

2. **Client Alliance Assessment:**
 * **Affective Bond (Score 1-5):** How well did the therapist foster an emotional connection, trust, and empathy? (1 = Very Poor, 5 = Very Strong)

3. **CTRS-Based Assessment (0-6 each, use integer scores):**
 * **Collaboration (0-6):** [criteria...]
 * **Guided Discovery (0-6):** [criteria...]
 * **Focus (0-6):** [criteria...]
 * **Strategy (0-6):** [criteria...]

[Output Format]
You MUST return the response in this EXACT JSON structure. Do not skip any fields.

{{
  "therapist_skills": {{
    "understanding": 0.0,
    "interpersonal_effectiveness": 0.0
  }},
  "client_alliance": {{
    "affective_bond": 0.0
  }},
  "ctrs": {{
    "collaboration": 0,
    "guided_discovery": 0,
    "focus": 0,
    "strategy": 0
  }},
  "reasoning": "Explain based on SPECIFIC TURN NUMBERS why this score was given. Highlight where the therapist missed or caught subtext.",
  "comparative_advantage": "Explain why this model is better or worse than a basic text-only therapist. If metadata was provided, did the therapist use it effectively?"
}}
"""
print("✅ Setup complete. Prompts and paths are defined.")

🚀 Initializing AI Evaluator...
OpenAI API Key loaded.
Evaluation Model set to: gpt-4o
Our Model results path: ../llms_therapist/therapist_simulation_results.jsonl
Baseline Model results path: ../llms_therapist_base/therapist_simulation_results_BASELINE.jsonl
✅ Setup complete. Prompts and paths are defined.


### Unused

In [2]:
# # Data Loading & Formatting Functions

# import json
# from pathlib import Path

# BASE = Path(r"C:\Luna-AI-Therapist\dissonance\craft_dialogue")

# # ชี้ไปที่ไฟล์ทั้งสามชุด
# all_files = [
#     * (BASE / "baseline").glob("dialogue_*_full_baseline.jsonl"),
#     * (BASE / "emotion").glob("dialogue_*_full_emotion_online.jsonl"),
#     * (BASE / "dissonance").glob("dialogue_*_full_dissonance_online.jsonl"),
# ]

# def load_full_dialogue_jsonl(path: Path) -> str:
#     """
#     อ่านไฟล์ dialogue_X_full_...jsonl (หนึ่งไฟล์ = 1 dialogue, 1 บรรทัด = 1 turn)
#     แล้วแปลงเป็นข้อความแบบ:

#     Client: ...
#     Therapist: ...
#     """
#     lines = []
#     with path.open("r", encoding="utf-8") as f:
#         for line in f:
#             if not line.strip():
#                 continue
#             rec = json.loads(line)
#             lines.append(f"CLIENT: {rec['client']}")
#             lines.append(f"THERAPIST: {rec['therapist']}")
#     return "\n".join(lines)

# rows = []
# for path in all_files:
#     name = path.name
#     if "baseline" in name:
#         method = "baseline"
#     elif "emotion" in name:
#         method = "emotion"
#     elif "dissonance" in name:
#         method = "dissonance"
#     else:
#         method = "unknown"

# def format_conversation_text(turns_list, is_baseline=False):
#     """
#     Converts a list of turns into a single 'CLIENT: ... THERAPIST: ...' string.
#     """
#     full_text = ""
#     # key สำหรับเคสเก่า (ai_evaluation เดิม)
#     therapist_key_old = "therapist_response_baseline" if is_baseline else "therapist_response"

#     for turn in turns_list:

#         # ฝั่ง client: รองรับทั้ง transcript (ไฟล์เก่า) และ client (ไฟล์ใหม่)
#         client_text = turn.get("transcript")
#         if client_text is None:
#             client_text = turn.get("client", "[missing transcript]")

#         # ฝั่ง therapist: รองรับทั้ง therapist_response* (ไฟล์เก่า) และ therapist (ไฟล์ใหม่)
#         therapist_text = turn.get(therapist_key_old)
#         if therapist_text is None:
#             therapist_text = turn.get("therapist", "[missing response]")
#         full_text += f"CLIENT: {client_text}\n\n"
#         full_text += f"THERAPIST: {therapist_text}\n\n"

#     return full_text.strip()

# print("✅ Data loading and formatting functions are defined.")

### Continued

In [3]:
# Evaluation Function (Calling GPT-4o)

def evaluate_session(session_text, max_retries=3):
    """
    Calls the GPT-4o API to get evaluation scores.
    Retries on failure.
    """
    formatted_user_prompt = USER_PROMPT.format(CONVERSATION_TEXT_GOES_HERE=session_text)
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": formatted_user_prompt}
    ]
    
    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model=EVALUATION_MODEL,
                messages=messages,
                temperature=0.0, # Crucial for objective scoring
                response_format={"type": "json_object"} # Force JSON output
            )
            
            response_content = completion.choices[0].message.content
            # Try to parse the JSON to ensure it's valid
            scores = json.loads(response_content)
            return scores # Success!
        
        except Exception as e:
            print(f"Warning: Attempt {attempt + 1}/{max_retries} failed. Error: {e}")
            if attempt < max_retries - 1:
                time.sleep(5) # Wait 5 seconds before retrying
            else:
                return {
                    "error": f"Failed after {max_retries} retries.",
                    "last_error": str(e)
                }
    return None # Should not be reached

print("✅ AI Evaluation function is defined.")

✅ AI Evaluation function is defined.


In [4]:
def format_conversation_text(turns_list, is_baseline=False):
    """
    Converts a list of turns into a single 'CLIENT: ... THERAPIST: ...' string.
    รองรับทั้งโครงไฟล์เก่า (transcript + therapist_response*) และไฟล์ใหม่ (client + therapist).
    """
    full_text = ""
    therapist_key_old = "therapist_response_baseline" if is_baseline else "therapist_response"

    for turn in turns_list:
        # client
        client_text = turn.get("transcript")
        if client_text is None:
            client_text = turn.get("client", "[missing transcript]")

        # therapist
        therapist_text = turn.get(therapist_key_old)
        if therapist_text is None:
            therapist_text = turn.get("therapist", "[missing response]")

        full_text += f"CLIENT: {client_text}\n\n"
        full_text += f"THERAPIST: {therapist_text}\n\n"

    return full_text.strip()

In [5]:
# Main evaluation loop

from pathlib import Path
import json
import re

print("--- Starting Evaluation Process (4 methods, multiple dialogues) ---")

BASE = Path(r"C:\Luna-AI-Therapist\dissonance\craft_dialogue")

METHOD_DIRS = {
    "baseline": BASE / "baseline" / "baseline_outputs",
    "emotion": BASE / "emotion" / "emotion_outputs",
    "multimodal": BASE / "multimodal" / "multimodal_outputs",
    "dissonance": BASE / "dissonance" / "dissonance_outputs",
}

def extract_dialogue_id(path: Path):
    m = re.search(r"dialogue_(\d+)_", path.name)
    return int(m.group(1)) if m else None

def load_dialogue_folder(folder: Path, pattern: str, is_baseline: bool) -> dict:
    """
    อ่านหลายไฟล์ในโฟลเดอร์ แล้วคืน dict:
    {
        1: [turns...],
        2: [turns...],
        ...
    }
    """
    dialogues = {}

    file_list = sorted(folder.glob(pattern))
    if not file_list:
        print(f"❌ ERROR: No files found in {folder} with pattern: {pattern}")
        return {}

    for path in file_list:
        dialogue_id = extract_dialogue_id(path)
        if dialogue_id is None:
            print(f"⚠️ Skipping file with unrecognized name: {path.name}")
            continue

        turns = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                rec = json.loads(line)
                turns.append({
                    "transcript": rec["client"],
                    ("therapist_response_baseline" if is_baseline else "therapist_response"): rec["therapist"],
                })

        dialogues[dialogue_id] = turns

    print(f"✅ Loaded {len(dialogues)} dialogues from {folder}")
    return dialogues

# โหลดทั้ง 4 methods
baseline_dialogues = load_dialogue_folder(
    METHOD_DIRS["baseline"],
    "dialogue_*_full_baseline.jsonl",
    is_baseline=True,
)

emotion_dialogues = load_dialogue_folder(
    METHOD_DIRS["emotion"],
    "dialogue_*_full_emotion_online*.jsonl",
    is_baseline=False,
)

dissonance_dialogues = load_dialogue_folder(
    METHOD_DIRS["dissonance"],
    "dialogue_*_full_dissonance_online*.jsonl",
    is_baseline=False,
)

multimodal_dialogues = load_dialogue_folder(
    METHOD_DIRS["multimodal"],
    "dialogue_*_full_multimodal*.jsonl",
    is_baseline=False,
)

# lists สำหรับเก็บผล
baseline_eval_scores = []
emotion_eval_scores = []
multimodal_eval_scores = []
dissonance_eval_scores = []

# หา dialogue_id ที่มีครบทุก method
dialogue_ids = sorted(
    set(baseline_dialogues.keys())
    & set(emotion_dialogues.keys())
    & set(multimodal_dialogues.keys())
    & set(dissonance_dialogues.keys())
)

if not baseline_dialogues or not emotion_dialogues or not multimodal_dialogues or not dissonance_dialogues:
    print("❌ ERROR: Cannot start evaluation. One or more input folders failed to load.")
elif not dialogue_ids:
    print("❌ ERROR: No shared dialogue IDs found across all 4 methods.")
else:
    print(f"Loaded shared dialogue IDs: {dialogue_ids}")

    for dialogue_id in tqdm(dialogue_ids, desc="Evaluating Dialogues"):

        # --- Baseline ---
        print(f"\nEvaluating BASELINE Dialogue {dialogue_id}...")
        session_text = format_conversation_text(baseline_dialogues[dialogue_id], is_baseline=True)
        scores = evaluate_session(session_text)
        scores["dialogue_id"] = dialogue_id
        scores["method"] = "baseline"
        baseline_eval_scores.append(scores)
        u = scores.get("therapist_skills", {}).get("understanding")
        print(f"  -> Done. Score (Understanding): {u}")

        # --- Emotion (text-only) ---
        print(f"Evaluating EMOTION (text-only) Dialogue {dialogue_id}...")
        session_text = format_conversation_text(emotion_dialogues[dialogue_id], is_baseline=False)
        scores = evaluate_session(session_text)
        scores["dialogue_id"] = dialogue_id
        scores["method"] = "emotion_text_only"
        emotion_eval_scores.append(scores)
        u = scores.get("therapist_skills", {}).get("understanding")
        print(f"  -> Done. Score (Understanding): {u}")

        # --- Dissonance ---
        print(f"Evaluating DISSONANCE Dialogue {dialogue_id}...")
        session_text = format_conversation_text(dissonance_dialogues[dialogue_id], is_baseline=False)
        scores = evaluate_session(session_text)
        scores["dialogue_id"] = dialogue_id
        scores["method"] = "dissonance"
        dissonance_eval_scores.append(scores)
        u = scores.get("therapist_skills", {}).get("understanding")
        print(f"  -> Done. Score (Understanding): {u}")

        # --- Multimodal (Vocal-Aware) ---
        print(f"Evaluating MULTIMODAL (Vocal-Aware) Dialogue {dialogue_id}...")
        session_text = format_conversation_text(multimodal_dialogues[dialogue_id], is_baseline=False)
        scores = evaluate_session(session_text)
        scores["dialogue_id"] = dialogue_id
        scores["method"] = "multimodal_vocal_aware"
        multimodal_eval_scores.append(scores)
        u = scores.get("therapist_skills", {}).get("understanding")
        print(f"  -> Done. Score (Understanding): {u}")

    print("\n🎉 --- Evaluation Process Complete! ---")

--- Starting Evaluation Process (4 methods, multiple dialogues) ---
✅ Loaded 10 dialogues from C:\Luna-AI-Therapist\dissonance\craft_dialogue\baseline\baseline_outputs
✅ Loaded 10 dialogues from C:\Luna-AI-Therapist\dissonance\craft_dialogue\emotion\emotion_outputs
✅ Loaded 10 dialogues from C:\Luna-AI-Therapist\dissonance\craft_dialogue\dissonance\dissonance_outputs
✅ Loaded 10 dialogues from C:\Luna-AI-Therapist\dissonance\craft_dialogue\multimodal\multimodal_outputs
Loaded shared dialogue IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


Evaluating Dialogues:   0%|          | 0/10 [00:00<?, ?it/s]


Evaluating BASELINE Dialogue 1...
  -> Done. Score (Understanding): 4.0
Evaluating EMOTION (text-only) Dialogue 1...
  -> Done. Score (Understanding): 4.0
Evaluating DISSONANCE Dialogue 1...
  -> Done. Score (Understanding): 5.0
Evaluating MULTIMODAL (Vocal-Aware) Dialogue 1...
  -> Done. Score (Understanding): 4.0

Evaluating BASELINE Dialogue 2...
  -> Done. Score (Understanding): 4.0
Evaluating EMOTION (text-only) Dialogue 2...
  -> Done. Score (Understanding): 4.0
Evaluating DISSONANCE Dialogue 2...
  -> Done. Score (Understanding): 5.0
Evaluating MULTIMODAL (Vocal-Aware) Dialogue 2...
  -> Done. Score (Understanding): 4.0

Evaluating BASELINE Dialogue 3...
  -> Done. Score (Understanding): 4.0
Evaluating EMOTION (text-only) Dialogue 3...
  -> Done. Score (Understanding): 5.0
Evaluating DISSONANCE Dialogue 3...
  -> Done. Score (Understanding): 5.0
Evaluating MULTIMODAL (Vocal-Aware) Dialogue 3...
  -> Done. Score (Understanding): 4.0

Evaluating BASELINE Dialogue 4...
  -> Done. 

In [6]:
# Save Results

from pathlib import Path
import json

OUTPUT_DIR = Path("evaluation_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Define output filenames
BASELINE_MODEL_SCORE_FILE   = OUTPUT_DIR / "ai_evaluation_results_BASELINE.jsonl"
EMOTION_MODEL_SCORE_FILE    = OUTPUT_DIR / "ai_evaluation_results_EMOTION_TEXT_ONLY.jsonl"
MULTIMODAL_MODEL_SCORE_FILE = OUTPUT_DIR / "ai_evaluation_results_MULTIMODAL_VOCAL_AWARE.jsonl"
DISSONANCE_MODEL_SCORE_FILE = OUTPUT_DIR / "ai_evaluation_results_DISSONANCE.jsonl"

# --- Save Baseline Model Scores ---
try:
    with open(BASELINE_MODEL_SCORE_FILE, 'w', encoding='utf-8') as f:
        for entry in baseline_eval_scores:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    print(f"✅ Successfully saved 'Baseline' evaluation scores to '{BASELINE_MODEL_SCORE_FILE}'")
except Exception as e:
    print(f"❌ Error saving 'Baseline' scores: {e}")

# --- Save Emotion (text-only) Scores ---
try:
    with open(EMOTION_MODEL_SCORE_FILE, 'w', encoding='utf-8') as f:
        for entry in emotion_eval_scores:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    print(f"✅ Successfully saved 'Emotion (text-only)' evaluation scores to '{EMOTION_MODEL_SCORE_FILE}'")
except Exception as e:
    print(f"❌ Error saving 'Emotion (text-only)' scores: {e}")

# --- Save Dissonance Model Scores ---
try:
    with open(DISSONANCE_MODEL_SCORE_FILE, 'w', encoding='utf-8') as f:
        for entry in dissonance_eval_scores:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    print(f"✅ Successfully saved 'Dissonance' evaluation scores to '{DISSONANCE_MODEL_SCORE_FILE}'")
except Exception as e:
    print(f"❌ Error saving 'Dissonance' scores: {e}")

# --- Save Multimodal (Vocal-Aware) Scores ---
try:
    with open(MULTIMODAL_MODEL_SCORE_FILE, 'w', encoding='utf-8') as f:
        for entry in multimodal_eval_scores:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    print(f"✅ Successfully saved 'Multimodal (Vocal-Aware)' evaluation scores to '{MULTIMODAL_MODEL_SCORE_FILE}'")
except Exception as e:
    print(f"❌ Error saving 'Multimodal (Vocal-Aware)' scores: {e}")

✅ Successfully saved 'Baseline' evaluation scores to 'evaluation_outputs\ai_evaluation_results_BASELINE.jsonl'
✅ Successfully saved 'Emotion (text-only)' evaluation scores to 'evaluation_outputs\ai_evaluation_results_EMOTION_TEXT_ONLY.jsonl'
✅ Successfully saved 'Dissonance' evaluation scores to 'evaluation_outputs\ai_evaluation_results_DISSONANCE.jsonl'
✅ Successfully saved 'Multimodal (Vocal-Aware)' evaluation scores to 'evaluation_outputs\ai_evaluation_results_MULTIMODAL_VOCAL_AWARE.jsonl'


In [7]:
import numpy as np

# --- 1) ฟังก์ชันคำนวณค่าเฉลี่ย ---

def calculate_averages(score_list):
    df = pd.json_normalize(score_list)

    if 'error' in df.columns:
        df = df[df['error'].isnull()]

    if df.empty:
        return {
            "understanding_avg": 0,
            "interpersonal_effectiveness_avg": 0,
            "affective_bond_avg": 0,
            "ctrs_collab_avg": 0,
            "ctrs_guided_avg": 0,
            "ctrs_focus_avg": 0,
            "ctrs_strategy_avg": 0,
            "count": 0,
        }

    averages = {
        "understanding_avg": df['therapist_skills.understanding'].mean(),
        "interpersonal_effectiveness_avg": df['therapist_skills.interpersonal_effectiveness'].mean(),
        "affective_bond_avg": df['client_alliance.affective_bond'].mean(),
        "ctrs_collab_avg": df['ctrs.collaboration'].mean(),
        "ctrs_guided_avg": df['ctrs.guided_discovery'].mean(),
        "ctrs_focus_avg": df['ctrs.focus'].mean(),
        "ctrs_strategy_avg": df['ctrs.strategy'].mean(),
        "count": len(df),
    }
    return averages

# --- 2) คำนวณค่าเฉลี่ยแยก 4 methods ---

baseline_avg   = calculate_averages(baseline_eval_scores)
emotion_avg    = calculate_averages(emotion_eval_scores)
multimodal_avg = calculate_averages(multimodal_eval_scores)
dissonance_avg = calculate_averages(dissonance_eval_scores)

num_dialogues = len(baseline_eval_scores)  # ถ้าทุก method มีจำนวน dialogue เท่ากัน

# --- 3) สร้างตารางสรุป Methods เป็น rows, Metrics เป็น columns ---

summary_data = {
    "Method": [
        "Baseline",
        "Emotion (Text-Only)",
        "Multimodal (Vocal-Aware)",
        "Dissonance-Aware",
    ],
    "Understanding (0-6)": [
        f"{baseline_avg['understanding_avg']:.2f}",
        f"{emotion_avg['understanding_avg']:.2f}",
        f"{multimodal_avg['understanding_avg']:.2f}",
        f"{dissonance_avg['understanding_avg']:.2f}",
    ],
    "Interpersonal (0-6)": [
        f"{baseline_avg['interpersonal_effectiveness_avg']:.2f}",
        f"{emotion_avg['interpersonal_effectiveness_avg']:.2f}",
        f"{multimodal_avg['interpersonal_effectiveness_avg']:.2f}",
        f"{dissonance_avg['interpersonal_effectiveness_avg']:.2f}",
    ],
    "Affective Bond (1-5)": [
        f"{baseline_avg['affective_bond_avg']:.2f}",
        f"{emotion_avg['affective_bond_avg']:.2f}",
        f"{multimodal_avg['affective_bond_avg']:.2f}",
        f"{dissonance_avg['affective_bond_avg']:.2f}",
    ],
    "CTRS Collab (0-6)": [
        f"{baseline_avg['ctrs_collab_avg']:.2f}",
        f"{emotion_avg['ctrs_collab_avg']:.2f}",
        f"{multimodal_avg['ctrs_collab_avg']:.2f}",
        f"{dissonance_avg['ctrs_collab_avg']:.2f}",
    ],
    "CTRS Guided (0-6)": [
        f"{baseline_avg['ctrs_guided_avg']:.2f}",
        f"{emotion_avg['ctrs_guided_avg']:.2f}",
        f"{multimodal_avg['ctrs_guided_avg']:.2f}",
        f"{dissonance_avg['ctrs_guided_avg']:.2f}",
    ],
    "CTRS Focus (0-6)": [
        f"{baseline_avg['ctrs_focus_avg']:.2f}",
        f"{emotion_avg['ctrs_focus_avg']:.2f}",
        f"{multimodal_avg['ctrs_focus_avg']:.2f}",
        f"{dissonance_avg['ctrs_focus_avg']:.2f}",
    ],
    "CTRS Strategy (0-6)": [
        f"{baseline_avg['ctrs_strategy_avg']:.2f}",
        f"{emotion_avg['ctrs_strategy_avg']:.2f}",
        f"{multimodal_avg['ctrs_strategy_avg']:.2f}",
        f"{dissonance_avg['ctrs_strategy_avg']:.2f}",
    ],
    "Successful": [
        f"{baseline_avg['count']} / {num_dialogues}",
        f"{emotion_avg['count']} / {num_dialogues}",
        f"{multimodal_avg['count']} / {num_dialogues}",
        f"{dissonance_avg['count']} / {num_dialogues}",
    ],
}

# --- 4) แสดงผล Reasoning และ Comparative Advantage แยกตาม Method ---

def print_qualitative_feedback(method_name, score_list):
    print(f"\n--- {method_name} Qualitative Feedback ---")
    for i, score in enumerate(score_list):
        # ดึงข้อมูลจาก dictionary (ถ้าไม่มีให้แสดง 'N/A')
        reasoning = score.get('reasoning', 'N/A')
        advantage = score.get('comparative_advantage', 'N/A')
        
        print(f"Dialogue {i+1}:")
        print(f"  > Reasoning: {reasoning}")
        print(f"  > Comparative Advantage: {advantage}")
    print("-" * 50)

summary_df = pd.DataFrame(summary_data)

print("Average Scores Across All Evaluated Dialogues:")
display(summary_df)

# เรียกใช้งาน summary reasoning
print_qualitative_feedback("Baseline", baseline_eval_scores)
print_qualitative_feedback("Emotion (Text-Only)", emotion_eval_scores)
print_qualitative_feedback("Multimodal (Vocal-Aware)", multimodal_eval_scores)
print_qualitative_feedback("Dissonance-Aware", dissonance_eval_scores)

Average Scores Across All Evaluated Dialogues:


,Method,Understanding (0-6),Interpersonal (0-6),Affective Bond (1-5),CTRS Collab (0-6),CTRS Guided (0-6),CTRS Focus (0-6),CTRS Strategy (0-6),Successful
0,Baseline,4.00,4.40,3.50,3.20,2.60,3.10,2.60,10 / 10
1,Emotion (Text-Only),4.10,5.00,4.00,4.00,3.10,3.50,3.10,10 / 10
2,Multimodal (Vocal-Aware),4.20,4.90,3.90,3.70,2.90,3.50,2.90,10 / 10
3,Dissonance-Aware,4.80,5.00,4.00,4.10,3.80,4.10,3.20,10 / 10



--- Baseline Qualitative Feedback ---
Dialogue 1:
  > Reasoning: The therapist demonstrated a good understanding of the client's expressed concerns about work-related stress and anxiety (Turns 2, 4, 6). However, the therapist often repeated the client's statements without delving deeper into the emotional subtext, such as the client's underlying fear of failure and guilt (Turns 8, 10, 12). The therapist maintained a supportive relationship, validating the client's feelings and encouraging exploration of potential solutions (Turns 14, 16). However, the therapist missed opportunities to explore the client's latent concerns, such as the fear of disappointing others and the internalized pressure to be perfect. The therapist's approach was supportive but lacked depth in uncovering the client's deeper emotional struggles.
  > Comparative Advantage: This model is slightly better than a basic text-only therapist because it maintains a strong therapeutic alliance and provides consistent valida